In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

import plotly.graph_objects as go

data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(
    data_url,
    sep=r"\s+",
    skiprows=22,
    header=None
)
data = np.hstack([
    raw_df.values[::2, :],
    raw_df.values[1::2, :2]
])

target = raw_df.values[1::2, 2]
feature_names = [
    "CRIM", "ZN", "INDUS", "CHAS", "NOX",
    "RM", "AGE", "DIS", "RAD", "TAX",
    "PTRATIO", "B", "LSTAT"
]
df = pd.DataFrame(data, columns=feature_names)
df["PRICE"] = target
print(df.head())
print("\nShape of Dataset:")
print(df.shape)
print("\nNull Values:")
print(df.isnull().sum())
print("\nDataset Info:")
print(df.info())
print("\nStatistical Summary:")
print(df.describe())
plt.figure(figsize=(8,5))
sns.histplot(df["PRICE"], kde=True)
plt.title("Price Distribution")
plt.show()

plt.figure(figsize=(8,5))
sns.boxplot(x=df["PRICE"])
plt.title("Boxplot of Price")
plt.show()
correlation = df.corr()
print("\nCorrelation with PRICE:")
print(correlation["PRICE"])
plt.figure(figsize=(15,12))
sns.heatmap(correlation, annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()
features = ['LSTAT', 'RM', 'PTRATIO']
plt.figure(figsize=(18,5))

for i, col in enumerate(features):
    plt.subplot(1, 3, i+1)
    plt.scatter(df[col], df["PRICE"])
    plt.xlabel(col)
    plt.ylabel("PRICE")
    plt.title(f"{col} vs PRICE")

plt.tight_layout()
plt.show()
X = df.iloc[:, :-1]
y = df["PRICE"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("\n===== Linear Regression Results =====")
print("MSE :", mse_lr)
print("RMSE:", rmse_lr)
print("MAE :", mae_lr)
print("R2 Score:", r2_lr)

model = Sequential()

model.add(Input(shape=(13,)))

model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(1))
model.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mae']
)
model.summary()

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    validation_split=0.05,
    verbose=1
)

fig = go.Figure()

fig.add_trace(go.Scatter(
    y=history.history['loss'],
    mode='lines',
    name='Training Loss'
))

fig.add_trace(go.Scatter(
    y=history.history['val_loss'],
    mode='lines',
    name='Validation Loss'
))

fig.update_layout(
    title='Model Loss',
    xaxis_title='Epoch',
    yaxis_title='Loss',
    height=500,
    width=700
)
fig.show()

fig = go.Figure()

fig.add_trace(go.Scatter(
    y=history.history['mae'],
    mode='lines',
    name='Training MAE'
))

fig.add_trace(go.Scatter(
    y=history.history['val_mae'],
    mode='lines',
    name='Validation MAE'
))

fig.update_layout(
    title='Model MAE',
    xaxis_title='Epoch',
    yaxis_title='MAE',
    height=500,
    width=700
)
fig.show()

y_pred_nn = model.predict(X_test)

mse_nn = mean_squared_error(y_test, y_pred_nn)
rmse_nn = np.sqrt(mse_nn)
mae_nn = mean_absolute_error(y_test, y_pred_nn)
r2_nn = r2_score(y_test, y_pred_nn)

print("\n===== Neural Network Results =====")
print("MSE :", mse_nn)
print("RMSE:", rmse_nn)
print("MAE :", mae_nn)
print("R2 Score:", r2_nn)

new_data = np.array([[
    0.1, 10.0, 5.0, 0, 0.4,
    6.0, 50, 6.0, 1, 400,
    20, 300, 10
]])

prediction = model.predict(new_data_scaled)

print("\nPredicted House Price:")
print(prediction)

